In [30]:
import os
import pandas as pd 
import re

In [31]:
# Config
DATASET = 'Mouse-2023' 
RAW_DIR = os.path.join(DATASET, 'raw-data')
PROCESSED_DIR = os.path.join(DATASET, 'processed-data')

**Make Counts Matrix**

In [58]:
counts_path = os.path.join(RAW_DIR, "gene_counts_matrix.tsv")
counts_df = pd.read_csv(counts_path, sep="\t", index_col=0)
counts_df.shape
# counts are in genes x samples format for DESeq2, need to transpose
counts_df = counts_df.T
counts_df.shape
# now counts are samples x genes (12 samples x 37991 genes)
counts_df.head()
# Add index name (sample)
counts_df.columns.name = None
counts_df.index.name = "sample"
# Save processed counts to a new file 
counts_df = counts_df.reset_index()
processed_counts_path = os.path.join(PROCESSED_DIR, "counts.csv")
counts_df.to_csv(processed_counts_path, index=False)

**Make Metadata**

In [59]:
# Extract sample IDs from the index of the counts dataframe
counts_df = pd.read_csv(processed_counts_path, index_col='sample')
sample_ids = counts_df.index.tolist()

# Add treatment column
def sample_map(n):
  if 142 <= n <= 145: return "control"
  elif 146 <= n <= 149: return "CFA"
  elif 150 <= n <= 153: return "CFB"

metadata = []
for sid in sample_ids: 
  match = re.search(r"(\d+)$", sid)
  n = int(match.group(1))
  treatment = sample_map(n)
  metadata.append({
    "sample_ID": sid,
    "treatment": treatment})
  
metadata_df = pd.DataFrame(metadata)

# Add replicate column
metadata_df['replicate'] = metadata_df.groupby('treatment').cumcount() + 1

# Add intuitive sample name 
metadata_df['sample'] = metadata_df['treatment'] + "_rep" + metadata_df['replicate'].astype(str)

# Reorder columns and save
cols = ['sample', 'sample_ID', 'treatment', 'replicate']
metadata_df = metadata_df[cols]
metadata_path = os.path.join(PROCESSED_DIR, "metadata.csv")
metadata_df.to_csv(metadata_path, index=False)

# Replace sample names in counts file 
counts_df = pd.read_csv(processed_counts_path, index_col=None)
counts_df['sample'] = metadata_df['sample']
counts_df.to_csv(processed_counts_path, index=False)

**Make QC Stats**

In [32]:
# Config 
mapping_file = os.path.join(RAW_DIR, "STAR_mapping_stats.csv")
counts_file = os.path.join(PROCESSED_DIR, "counts.csv")
qc_out_path = os.path.join(PROCESSED_DIR, "qc_metrics.csv")

reads_threshold = 1.5e7
mapping_threshold = 0.5
rRNA_threshold = 0.15


In [33]:
# Read the files
mapping_df = pd.read_csv(mapping_file, sep=",", header=0)
counts_df = pd.read_csv(counts_file, index_col='sample')

mapping_df.shape # There are 12 samples and 5 mapping stats
mapping_df['Sample'] = counts_df.index # Clean up sample names
mapping_df = mapping_df.set_index('Sample') # Set sample names as index for easier merging with counts_df later


In [34]:
mapping_df.head()

,Number of input reads,Uniquely mapped reads number,Uniquely mapped reads %,Number of reads mapped to multiple loci,% of reads mapped to multiple loci
Sample,,,,,
control_rep1,28250055,25615057,90.67%,2136935,7.56%
control_rep2,27553572,24840617,90.15%,2184201,7.93%
control_rep3,29733702,27049908,90.97%,2156935,7.25%
control_rep4,30245730,27567448,91.14%,2185175,7.22%
CFA_rep1,29911786,26893945,89.91%,2389618,7.99%


In [36]:
qc_df = pd.DataFrame(index=counts_df.index)

# Mapping stats
qc_df['total_reads'] = mapping_df['Number of input reads'].astype(int)
qc_df['mapping_rate'] = mapping_df['Uniquely mapped reads %'].str.rstrip('%').astype(float) / 100
qc_df['n_detected_genes'] = (counts_df > 0).sum(axis=1)
qc_df['multimapping_rate'] = mapping_df['% of reads mapped to multiple loci'].str.rstrip('%').astype(float) / 100

MAD_thresholds = {}
for stat in qc_df.columns:
  median = qc_df[stat].median()
  mad = (qc_df[stat] - median).abs().median() * 1.4826
  lower = median - (2 * mad)
  higher = median + (2 * mad)
  MAD_thresholds[stat] = (lower, higher)
thresholds_df = pd.DataFrame(MAD_thresholds, index=['mad_lower', 'mad_upper']).T

qc_df.head()

,total_reads,mapping_rate,n_detected_genes,multimapping_rate
sample,,,,
control_rep1,28250055,0.9067,19729,0.0756
control_rep2,27553572,0.9015,19765,0.0793
control_rep3,29733702,0.9097,19599,0.0725
control_rep4,30245730,0.9114,19987,0.0722
CFA_rep1,29911786,0.8991,19898,0.0799


In [37]:
for sample in qc_df.index:
  # Absolute thresholds
  a_reasons = []
  if qc_df.loc[sample, 'total_reads'] < reads_threshold: a_reasons.append('low_reads')
  if qc_df.loc[sample, 'mapping_rate'] < mapping_threshold: a_reasons.append('low_mapping_rate')
  if qc_df.loc[sample, 'multimapping_rate'] > rRNA_threshold: a_reasons.append('high_multimapping_rate')
  if a_reasons ==[]: 
    qc_df.loc[sample, 'absolute_thresholds'] = 'Pass'
  else: 
    qc_df.loc[sample, 'absolute_thresholds'] = ','.join(a_reasons)
  # Relative thresholds 
  r_reasons = []
  if qc_df.loc[sample, 'total_reads'] < thresholds_df.loc['total_reads', 'mad_lower']: r_reasons.append('low_reads')
  if qc_df.loc[sample, 'mapping_rate'] < thresholds_df.loc['mapping_rate', 'mad_lower']: r_reasons.append('low_mapping_rate')
  if qc_df.loc[sample, 'multimapping_rate'] > thresholds_df.loc['multimapping_rate', 'mad_upper']: r_reasons.append('high_multimapping_rate')
  if qc_df.loc[sample, 'n_detected_genes'] < thresholds_df.loc['n_detected_genes', 'mad_lower']: r_reasons.append('low_n_detected_genes')
  if r_reasons ==[]:
    qc_df.loc[sample, 'relative_thresholds'] = 'Pass'
  else:
    qc_df.loc[sample, 'relative_thresholds'] = ','.join(r_reasons)
  
qc_df.to_csv(qc_out_path)

In [38]:
qc_df.head()

,total_reads,mapping_rate,n_detected_genes,multimapping_rate,absolute_thresholds,relative_thresholds
sample,,,,,,
control_rep1,28250055,0.9067,19729,0.0756,Pass,Pass
control_rep2,27553572,0.9015,19765,0.0793,Pass,Pass
control_rep3,29733702,0.9097,19599,0.0725,Pass,Pass
control_rep4,30245730,0.9114,19987,0.0722,Pass,Pass
CFA_rep1,29911786,0.8991,19898,0.0799,Pass,Pass
